In [ ]:
import numpy as np
from gen.gen_interpFunction import interpLagGLQ
from gen.gen_gaussQuadCalc import gLQ
from gen.gen_utilities import permutation_symbol, rotTensor, rotVector
from gen.gen_mesh1D import mesh1DLGL
import pandas as pd

NNPEL=2
NEL = 5
DOFPN = 6
NGQP = 1
interp, interpDiff = interpLagGLQ(NNPEL, NGQP)
moment= 2*4 * np.pi
gauss = gLQ(NGQP)
ECON, elemGlobalCoord = mesh1DLGL(1, NEL, NNPEL)
print('ECON=',ECON)
print('elemGlobalCoord=',elemGlobalCoord)
print('interp=',interp)
print('interpDiff=',interpDiff)
print('gauss Points=', gLQ(NGQP)['points'])
print('gauss Weights=', gLQ(NGQP)['weights'])
permutation = permutation_symbol()
Jacobian = np.dot(np.array([[0,0.2],[0.2,0.4],[0.4,0.6],[0.6,0.8],[0.8,1.0]]), interpDiff)
print('Jacobian=',Jacobian)
phi_o = np.array([[0.0,0.0,0.0],
        [0.0,0.0,0.2],
        [0.0,0.0,0.4],
        [0.0,0.0,0.6000000000000001],
        [0.0,0.0,0.8],
        [0.0,0.0,1.0]])

matModF = np.diag([2985728000000,2985728000000,2299017])
matModM = np.diag([1.9999999992,1.9999999992,1.5238088])

nu = np.array([[0,0,0],[0,0,0],[0,0,0],[0,0,0],[0,0,0],[0,0,0]],dtype=np.float64)
netRotVec = np.array([[0,0,0],[0,0,0],[0,0,0],[0,0,0],[0,0,0],[0,0,0]],dtype=np.float64)
prevOmega = np.array([[0,0,0],[0,0,0],[0,0,0],[0,0,0],[0,0,0],[0,0,0]],dtype=np.float64)
appForce = np.array([[0,0,0],[0,0,0],[0,0,0],[0,0,0],[0,0,0],[0,0,0]],dtype=np.float64)
appMoment = np.array([[0,0,0],[0,0,0],[0,0,0],[0,0,0],[0,0,0],[0,0,0]],dtype=np.float64)


ECON= [[0 1]
 [1 2]
 [2 3]
 [3 4]
 [4 5]]
elemGlobalCoord= [[0.  0.2]
 [0.2 0.4]
 [0.4 0.6]
 [0.6 0.8]
 [0.8 1. ]]
interp= [[0.5]
 [0.5]]
interpDiff= [[-0.5]
 [ 0.5]]
gauss Points= [0.]
gauss Weights= [2.]
Jacobian= [[0.1]
 [0.1]
 [0.1]
 [0.1]
 [0.1]]


In [21]:
iter = 0
# For element #1 and Gauss point #1
element_index = 4  # Indexing starts from 0
gauss_point_index = 0  # Indexing starts from 0
effeWt = Jacobian[element_index, gauss_point_index] * gauss['weights'][gauss_point_index]
print('effeWt=', effeWt)
S0 = interp[:, gauss_point_index]
S1 = interpDiff[:, gauss_point_index]/Jacobian[element_index, gauss_point_index]
print('S0=', S0)
print('S1=', S1)
outerS00 = np.outer(S0, S0)
outerS01 = np.outer(S0, S1)
outerS10 = np.outer(S1, S0)
outerS11 = np.outer(S1, S1)
print('outerS00=', outerS00)
print('outerS01=', outerS01)
print('outerS10=', outerS10)
print('outerS11=', outerS11)

dphi_oGP = np.dot(S1, phi_o[ECON[element_index], :])
print('dphi_oGP=', dphi_oGP)
nuGP =S1 @ nu[ECON[element_index], :]
print('nuGP=', nuGP)
netRotVecGP = S0 @ netRotVec[ECON[element_index], :]
print('netRotVecGP=', netRotVecGP)
netRotMatGP = rotTensor(netRotVecGP)
print('netRotMatGP=', netRotMatGP)
incRotMatGP = rotTensor(nuGP)
print('incRotMatGP=', incRotMatGP)
prevOmegaGP = S0 @ prevOmega[ECON[element_index], :]
print('prevOmegaGP=', prevOmegaGP)
normNu = np.linalg.norm(nuGP)
print('normNu=', normNu)
if normNu == 0:
    beta = 0
    print('Normnu=0, therefore beta=0')
xi = incRotMatGP @ prevOmegaGP
print('xi=', xi)
omegaGP = beta + xi
print('omegaGP=', omegaGP)
gammaGP = dphi_oGP - netRotMatGP @ np.array([0,0,1],dtype=np.float64)
print('gammaGP=', gammaGP)
matModFGP = netRotMatGP @ matModF @ netRotMatGP.T
print('matModFGP=', matModFGP)
matModMGP = netRotMatGP @ matModM @ netRotMatGP.T
print('matModMGP=', matModMGP)

forceGP = matModFGP @ gammaGP
print('forceGP=', forceGP)
momentGP = matModMGP @ omegaGP
print('momentGP=', momentGP)

appForceGP = S0 @ appForce[ECON[element_index], :]
print('appForceGP=', appForceGP)
appMomentGP = S0 @ appMoment[ECON[element_index], :]
print('appMomentGP=', appMomentGP)


effeWt= 0.19999999999999996
S0= [0.5 0.5]
S1= [-5.  5.]
outerS00= [[0.25 0.25]
 [0.25 0.25]]
outerS01= [[-2.5  2.5]
 [-2.5  2.5]]
outerS10= [[-2.5 -2.5]
 [ 2.5  2.5]]
outerS11= [[ 25. -25.]
 [-25.  25.]]
dphi_oGP= [0. 0. 1.]
nuGP= [0. 0. 0.]
netRotVecGP= [0. 0. 0.]
netRotMatGP= [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
incRotMatGP= [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
prevOmegaGP= [0. 0. 0.]
normNu= 0.0
Normnu=0, therefore beta=0
xi= [0. 0. 0.]
omegaGP= [0. 0. 0.]
gammaGP= [0. 0. 0.]
matModFGP= [[2.985728e+12 0.000000e+00 0.000000e+00]
 [0.000000e+00 2.985728e+12 0.000000e+00]
 [0.000000e+00 0.000000e+00 2.299017e+06]]
matModMGP= [[2.        0.        0.       ]
 [0.        2.        0.       ]
 [0.        0.        1.5238088]]
forceGP= [0. 0. 0.]
momentGP= [0. 0. 0.]
appForceGP= [0. 0. 0.]
appMomentGP= [ 0.         12.56637061  0.        ]


In [22]:
a=1
b=0

coeffSME = {}
coeffCVE = {}
coeffSME[f'K{a}{b}'] = matModFGP[a,b] *outerS11 * effeWt
coeffSME[f'K{a}{b+3}'] = (dphi_oGP @ permutation[:,b,:] @ matModFGP[a,:].T - forceGP[:] @ permutation[:,b,a]) * outerS10 * effeWt
coeffSME[f'K{a+3}{b}'] = (dphi_oGP @ permutation[:,a,:] @ matModFGP[b,:].T + forceGP[:] @ permutation[:,b,a])  * outerS01 * effeWt

coeffSME[f'K{a+3}{b+3}'] = ((matModMGP[a,b] * outerS11) + ((-momentGP @ permutation[:,b,a]) * outerS10 ) \
        + (-(dphi_oGP @ permutation[:,a,:] @ matModFGP @ permutation[:,b,:] @ dphi_oGP)+(dphi_oGP @ permutation[:,a,:] @ permutation[:,b,:] @ forceGP)) * outerS00) * effeWt

coeffCVE[f'F{a}'] = (appForceGP[a] * S0 - forceGP[a] * S1) * effeWt
coeffCVE[f'F{a+3}'] = (((permutation[a,:,:] @ forceGP) @ dphi_oGP + appMomentGP[a]) * S0 - momentGP[a] * S1) * effeWt

print(f'coeffSME[K{a}{b}] =', coeffSME[f'K{a}{b}'])
print(f'coeffSME[K{a}{b+3}] =', coeffSME[f'K{a}{b+3}'])
print(f'coeffSME[K{a+3}{b}] =', coeffSME[f'K{a+3}{b}'])
print(f'coeffSME[K{a+3}{b+3}] =', coeffSME[f'K{a+3}{b+3}'])
print(f'coeffCVE[F{a}] = ',coeffCVE[f'F{a}'])
print(f'coeffCVE[F{a+3}] = ',coeffCVE[f'F{a+3}'])

coeffSME[K10] = [[ 0. -0.]
 [-0.  0.]]
coeffSME[K13] = [[-1.492864e+12 -1.492864e+12]
 [ 1.492864e+12  1.492864e+12]]
coeffSME[K40] = [[ 1.492864e+12 -1.492864e+12]
 [ 1.492864e+12 -1.492864e+12]]
coeffSME[K43] = [[0. 0.]
 [0. 0.]]
coeffCVE[F1] =  [0. 0.]
coeffCVE[F4] =  [1.25663706 1.25663706]


In [4]:
permutation[:,0,:]

array([[ 0.,  0.,  0.],
       [ 0.,  0., -1.],
       [ 0.,  1.,  0.]])

In [13]:
ECON[4]

array([4, 5], dtype=int32)

array([ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.12568211,  0.        ,  0.        ,  0.        ,
        1.25682108,  0.        ,  0.50272171, -0.        ,  0.        ,
       -0.        ,  2.51357492,  0.        ,  1.13107118, -0.        ,
        0.        ,  0.        ,  3.76991978,  0.        ,  2.01069385,
       -0.        ,  0.        ,  0.        ,  5.02630696,  0.        ,
        3.14161053, -0.        ,  0.        ,  0.        ,  6.28285981,
        0.        ])

In [1]:
from gen.gen_utilities import *

In [30]:
c = np.array([7.28452E-06,1.45726E-05,2.22413E-05,2.66336E-05,2.70029E-05], dtype=np.float64)
p= np.array([-0.00823613,-0.0146393,-0.01921438,-0.02196657,-0.02288294], dtype=np.float64)
y=np.array([0, 0 , 0, 0, 0], dtype=np.float64)
for i in range(5):
    x = rotVector(rotTensor(np.array([0.0,c[i],0.0])) @ rotTensor(np.array([0.0,p[i],0.0])))
    print('i=',i,'Vec=',x)
    y[i] = x[1]
print(y)

i= 0 Vec= [ 0.         -0.00822885  0.        ]
i= 1 Vec= [ 0.         -0.01462473  0.        ]
i= 2 Vec= [ 0.         -0.01919214  0.        ]
i= 3 Vec= [ 0.         -0.02193994  0.        ]
i= 4 Vec= [ 0.         -0.02285594  0.        ]
[-0.00822885 -0.01462473 -0.01919214 -0.02193994 -0.02285594]


In [2]:
from gen.gen_utilities import *
import numpy as np

matF = np.array([[4.8E+13, 0, 0],[0,4.8E+13,0],[0,0,525000000]])
matM = np.array([[109374.93, 0, 0],[0,109374.93,0],[0,0,83332.8]])

vec= np.array([-0.003820215,
-0.010542636,
-0.01542251,
-0.018653371,
-0.020401279], dtype=np.float64)

gammaX = np.array([-6.218619217658594e-10,
1.9617582135145195e-08,
-1.1805728278649341e-08,
3.8799055859448095e-07,
1.9742836395233998e-07], dtype=np.float64)

gammaZ = np.array([8.00E-08,
 -4.042879935939325e-06,
4.1076495116421086e-06,
-8.405956163803907e-06,
4.342048642635987e-06], dtype=np.float64)

omega = np.array([-0.03819476,
-0.02903706,
-0.01979203,
-0.01250404,
-0.00493736], dtype=np.float64)

for i in range(5):
    x = rotTensor(np.array([0,vec[i],0]))
    matFGP = x @ matF @ x.T
    print('i=',i,'matFGP=',matFGP)
    matMGP = x @ matM @ x.T
    print('i=',i,'matMGP=',matMGP)
    gpF = matFGP @ np.array([gammaX[i],0,gammaZ[i]])
    gpM = matMGP @ np.array([0,omega[i],0])
    print('i=',i,'gpM=',gpM)
    print('i=',i,'gpF=',gpF)

i= 0 matFGP= [[4.79992995e+13 0.00000000e+00 1.83366530e+11]
 [0.00000000e+00 4.80000000e+13 0.00000000e+00]
 [1.83366530e+11 0.00000000e+00 1.22550298e+09]]
i= 0 matMGP= [[1.09374550e+05 0.00000000e+00 9.94855677e+01]
 [0.00000000e+00 1.09374930e+05 0.00000000e+00]
 [9.94855677e+01 0.00000000e+00 8.33331801e+04]]
i= 0 gpM= [    0.         -4177.54920137     0.        ]
i= 0 gpF= [-15179.61420178      0.            -15.98842475]
i= 1 matFGP= [[4.79946652e+13 0.00000000e+00 5.06003497e+11]
 [0.00000000e+00 4.80000000e+13 0.00000000e+00]
 [5.06003497e+11 0.00000000e+00 5.85980834e+09]]
i= 1 matMGP= [[109372.03559809      0.            274.53235387]
 [     0.         109374.93            0.        ]
 [   274.53235387      0.          83335.69440191]]
i= 1 gpM= [    0.         -3175.92640491     0.        ]
i= 1 gpF= [-1104172.10016347        0.           -13763.93638779]
i= 2 matFGP= [[4.79885840e+13 0.00000000e+00 7.40155004e+11]
 [0.00000000e+00 4.80000000e+13 0.00000000e+00]
 [7.401550

In [20]:
nu = np.array([0-0.001496763,
-0.004489565,
-0.007482297,
-0.010071716,
-0.011508134],dtype=np.float64)

nudiff = np.array([-0.014967628,
-0.014960391,
-0.014966931,
-0.010927261,
-0.00343692],dtype=np.float64)

for i in range(5):
    aNu = np.array([0,nu[i],0])
    # print('aNu=',aNu)
    aNudiff = np.array([0,nudiff[i],0])
    # print('aNuDiff=',aNudiff)
    normNu = np.linalg.norm(aNu)
    print('normNu=',normNu)
    termI = np.sin(normNu)/normNu * aNudiff
    # print('termI=',termI)
    termII = (1-np.sin(normNu)/normNu) * (np.dot(aNu, aNudiff)/normNu) * aNu/normNu
    # print('np.dot(aNu, aNudiff)=',np.dot(aNu, aNudiff))
    # print('termII=',termII)
    termIII = (2*(np.sin(0.5*normNu)/normNu)**2) * np.cross(aNu, aNudiff)
    # print('termIII=',termIII)

    omega = termI + termII + termIII
    print('beta=',omega)

normNu= 0.001496763
beta= [ 0.         -0.01496763  0.        ]
normNu= 0.004489565
beta= [ 0.         -0.01496039  0.        ]
normNu= 0.007482297
beta= [ 0.         -0.01496693  0.        ]
normNu= 0.010071716
beta= [ 0.         -0.01092726  0.        ]
normNu= 0.011508134
beta= [ 0.         -0.00343692  0.        ]
